<a href="https://colab.research.google.com/github/Tresorshingiro/Malaria-Diagnosis-CNN-Transfer-Learning/blob/main/Baseline_CNN__Model_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Malaria Diagnosis
This notebook is inspired by works of (Sivaramakrishnan Rajaraman  et al., 2018) and (Jason Brownlee, 2019). Acknowledge to NIH and Bangalor Hospital who make available this malaria dataset.

Malaria is an infectuous disease caused by parasites that are transmitted to people through the bites of infected female Anopheles mosquitoes.

The Malaria burden with some key figures:
<font color='red'>
* More than 219 million cases
* Over 430 000 deaths in 2017 (Mostly: children & pregnants)
* 80% in 15 countries of Africa & India
  </font>

![MalariaBurd](https://github.com/habiboulaye/ai-labs/blob/master/malaria-diagnosis/doc-images/MalariaBurden.png?raw=1)

The malaria diagnosis is performed using blood test:
* Collect patient blood smear
* Microscopic visualisation of the parasit

![MalariaDiag](https://github.com/habiboulaye/ai-labs/blob/master/malaria-diagnosis/doc-images/MalariaDiag.png?raw=1)
  
Main issues related to traditional diagnosis:
<font color='#ed7d31'>
* resource-constrained regions
* time needed and delays
* diagnosis accuracy and cost
</font>

The objective of this notebook is to apply modern deep learning techniques to perform medical image analysis for malaria diagnosis.

*This notebook is inspired by works of (Sivaramakrishnan Rajaraman  et al., 2018), (Adrian Rosebrock, 2018) and (Jason Brownlee, 2019)*

## Configuration

In [ ]:
#Mount the local drive project_forder
from google.colab import drive
drive.mount('/content/drive/')
!ls "/content/drive/My Drive/Colab Notebooks/10xDS/Projects/malaria-diagnosis/"

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
ls: cannot access '/content/drive/My Drive/Colab Notebooks/10xDS/Projects/malaria-diagnosis/': No such file or directory


In [ ]:
# Use GPU: Please check if the outpout is '/device:GPU:0'
import tensorflow as tf
print(tf.__version__)
tf.test.gpu_device_name()
#from tensorflow.python.client import device_lib
#device_lib.list_local_devices()

2.20.0


'/device:GPU:0'

## Populating namespaces

In [ ]:
# Importing basic libraries
import os
import random
import shutil
import tensorflow as tf
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.image import imread
%matplotlib inline

# Importing the Keras libraries and packages
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Convolution2D as Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

In [ ]:
# Define the useful paths for data accessibility
ai_project = '.' #"/content/drive/My Drive/Colab Notebooks/ai-labs/malaria-diagnosis"
cell_images_dir = os.path.join(ai_project,'cell_images')
training_path = os.path.join(ai_project,'train')
testing_path = os.path.join(ai_project,'test')

## Prepare DataSet

### *Download* DataSet

In [ ]:
# Download the data in the allocated google cloud-server. If already down, turn downloadData=False
downloadData = True
if downloadData == True:
  indrive = False
  if indrive == True:
    !wget https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip -P "/content/drive/My Drive/Colab Notebooks/ai-labs/malaria-diagnosis"
    !unzip "/content/drive/My Drive/Colab Notebooks/ai-labs/malaria-diagnosis/cell_images.zip" -d "/content/drive/My Drive/Colab Notebooks/ai-labs/malaria-diagnosis/"
    !ls "/content/drive/My Drive/Colab Notebooks/ai-labs/malaria-diagnosis"
  else: #incloud google server
    !rm -rf cell_images.*
    !wget https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip
    !unzip cell_images.zip >/dev/null 2>&1
    !ls

--2026-06-05 08:24:52--  https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip
Resolving data.lhncbc.nlm.nih.gov (data.lhncbc.nlm.nih.gov)... 18.161.6.61, 18.161.6.74, 18.161.6.26, ...
Connecting to data.lhncbc.nlm.nih.gov (data.lhncbc.nlm.nih.gov)|18.161.6.61|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 353452851 (337M) [application/zip]
Saving to: ‘cell_images.zip’

cell_images.zip     100%[===================>] 337.08M   291MB/s    in 1.2s    

2026-06-05 08:24:53 (291 MB/s) - ‘cell_images.zip’ saved [353452851/353452851]

cell_images  cell_images.zip  drive  sample_data


In [ ]:
!ls cell_images

import os

parasitized = len(os.listdir('cell_images/Parasitized'))
uninfected = len(os.listdir('cell_images/Uninfected'))

print("Parasitized:", parasitized)
print("Uninfected:", uninfected)

Parasitized  Uninfected
Parasitized: 13780
Uninfected: 13780


In [ ]:
from matplotlib import pyplot as plt
from matplotlib.image import imread
import os

plt.figure(figsize=(10,10))

folder = 'cell_images/Parasitized'

for i, filename in enumerate(os.listdir(folder)[:9]):
    img = imread(os.path.join(folder, filename))
    plt.subplot(3,3,i+1)
    plt.imshow(img)
    plt.axis('off')

plt.show()

In [ ]:
import tensorflow as tf

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "cell_images",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "cell_images",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(class_names)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

## Baseline CNN Model
Define a basic ConvNet defined with ConvLayer: Conv2D => MaxPooling2D followed by Flatten => Dense => Dense(output)

![ConvNet](https://github.com/habiboulaye/ai-labs/blob/master/malaria-diagnosis/doc-images/ConvNet.png?raw=1)


In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense
)

def build_model(dense_units=64, optimizer='adam'):

    model = Sequential([
        Input(shape=(128,128,3)),
        Rescaling(1./255),

        Conv2D(32, (3,3), activation='relu'),
        MaxPooling2D(),

        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(),

        Flatten(),

        Dense(dense_units, activation='relu'),

        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

Experiment 1 for Basline 10 Epochs

In [ ]:
baseline_model = build_model()

In [ ]:
 baseline_model = build_model()

history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Experiment 2 for 5 epochs

In [ ]:
model_e2 = build_model()

history2 = model_e2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Experiment 3 for 15 epochs

In [ ]:
model_e3 = build_model()

history3 = model_e3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

experiemnt 4 for Dense 128

In [ ]:
model_e4 = build_model(dense_units=128)

history4 = model_e4.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Experminet 5 for SDG

In [ ]:
model_e5 = build_model(optimizer='sgd')

history5 = model_e5.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Experiment 6 for BAtch size 16

In [ ]:
train_ds_16 = tf.keras.utils.image_dataset_from_directory(
    "cell_images",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(128,128),
    batch_size=16
)

val_ds_16 = tf.keras.utils.image_dataset_from_directory(
    "cell_images",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(128,128),
    batch_size=16
)
model_e6 = build_model()

history6 = model_e6.fit(
    train_ds_16,
    validation_data=val_ds_16,
    epochs=10
)

Experiment 7 for Batch size 64

In [ ]:
train_ds_64 = tf.keras.utils.image_dataset_from_directory(
    "cell_images",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(128,128),
    batch_size=64
)

val_ds_64 = tf.keras.utils.image_dataset_from_directory(
    "cell_images",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(128,128),
    batch_size=64
)

model_e7 = build_model()

history7 = model_e7.fit(
    train_ds_64,
    validation_data=val_ds_64,
    epochs=10
)

In [ ]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate(model, val_ds):

    y_true = []
    y_pred = []
    y_scores = []

    for images, labels in val_ds:
        preds = model.predict(images, verbose=0)

        y_true.extend(labels.numpy())
        y_pred.extend((preds > 0.5).astype(int).flatten())
        y_scores.extend(preds.flatten())

    print(classification_report(y_true, y_pred))

    return np.array(y_true), np.array(y_pred), np.array(y_scores)

In [ ]:
import pandas as pd
import numpy as np

results = []

models = {
    "E1_10epochs": baseline_model,
    "E2_5epochs": model_e2,
    "E3_15epochs": model_e3,
    "E4_Dense128": model_e4,
    "E5_SGD": model_e5,
    "E6_Batch16": model_e6,
    "E7_Batch64": model_e7
}

for name, model in models.items():

    y_true, y_pred, y_scores = evaluate(model, val_ds)

    results.append([
        name,
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred),
        recall_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Experiment",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score"
    ]
)

results_df

### Results and Analysis

Seven experiments were conducted on the baseline CNN model by changing the number of epochs, batch size, dense layer size, and optimizer.

The results showed that all experiments performed well, with accuracy ranging from 92% to 95%. The best result was achieved in Experiment 7 (Batch Size = 64), which obtained an accuracy of 94.9% and an F1-score of 95.1%.

Some experiments showed signs of overfitting. The training accuracy continued to increase, while the validation accuracy stayed around 93%–95%. This means the model learned the training data very well but did not improve much on unseen data.

Overall, the baseline CNN performed well in detecting malaria-infected cells and provided a strong foundation for comparing more advanced CNN and transfer learning models.


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5,5))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j],
                 ha='center',
                 va='center')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0,1],[0,1],'--')

plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.legend()
plt.show()

print("AUC:", roc_auc)

In [ ]:
y_true, y_pred, y_scores = evaluate(
    baseline_model,
    val_ds_64
)

In [ ]:

# Accuracy
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Loss
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()